## Objective

The 960×960 detector increased validation mAP@0.50:0.95 while producing nearly unchanged mAP@0.50. This suggests that higher input resolution may improve bounding-box localization. To demonstrate this, this notebook will determine:

1. Whether the 960×960 detector produces higher accuracy than the 640×640 detector.
2. Whether localization improvements are concentrated among smaller plates.
3. Which image characteristics are associated with poor localization.
4. Whether poor localization cases can be predicted using metadata features.

Each vehicle track contains multiple related video frames so tests will keep tracks grouped together.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from scipy.stats import kruskal, spearmanr, ttest_rel, wilcoxon

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ultralytics import YOLO

In [2]:
PROJECT_ROOT = Path.cwd().parent

YOLO_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "yolo_plate_detection"
)

METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "image_metadata.csv"
)

RUNS_DIR = PROJECT_ROOT / "runs" / "detect"

WEIGHTS_640 = (
    RUNS_DIR
    / "yolo26n_640_baseline"
    / "weights"
    / "best.pt"
)

WEIGHTS_960 = (
    RUNS_DIR
    / "yolo26n_960"
    / "weights"
    / "best.pt"
)

VAL_IMAGE_DIR = YOLO_DIR / "images" / "val"

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "prediction_analysis.csv"
)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

In [3]:
# From 02_model.ipynb results
reported_metrics = pd.DataFrame(
    {
        "640": {
            "precision": 0.960581,
            "recall": 0.974727,
            "mAP50": 0.992213,
            "mAP50-95": 0.737371,
        },
        "960": {
            "precision": 0.942881,
            "recall": 0.967778,
            "mAP50": 0.989613,
            "mAP50-95": 0.780363,
        },
    }
)

reported_metrics["difference"] = (
    reported_metrics["960"] - reported_metrics["640"]
)

reported_metrics

,640,960,difference
precision,0.960581,0.942881,-0.017700
recall,0.974727,0.967778,-0.006949
mAP50,0.992213,0.989613,-0.002600
mAP50-95,0.737371,0.780363,0.042992


In [4]:
# Load validation metadata
metadata = pd.read_csv(METADATA_PATH)

validation_metadata = (
    metadata.loc[metadata["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

print("Validation rows:", len(validation_metadata))
print("Validation tracks:", validation_metadata["track_id"].nunique())
print("Duplicate image names:", validation_metadata["image_name"].duplicated().sum())

validation_metadata.head()

Validation rows: 900
Validation tracks: 30
Duplicate image names: 0


,split,track_id,frame_number,image_name,image_path,annotation_path,image_width,image_height,camera,vehicle_type,...,plate_brightness,plate_contrast,plate_blur_score,corners_valid,plate_box_valid,vehicle_box_valid,plate_inside_vehicle,character_count_matches,character_boxes_valid,annotation_valid
0,validation,track0061,1,track0061[01].png,data/raw/ufpr_alpr/validation/track0061/track0...,data/raw/ufpr_alpr/validation/track0061/track0...,1920,1080,GoPro Hero4 Silver,car,...,210.369907,43.041450,5543.694393,True,True,True,True,True,True,True
1,validation,track0061,2,track0061[02].png,data/raw/ufpr_alpr/validation/track0061/track0...,data/raw/ufpr_alpr/validation/track0061/track0...,1920,1080,GoPro Hero4 Silver,car,...,209.325736,42.019851,5536.883216,True,True,True,True,True,True,True
2,validation,track0061,3,track0061[03].png,data/raw/ufpr_alpr/validation/track0061/track0...,data/raw/ufpr_alpr/validation/track0061/track0...,1920,1080,GoPro Hero4 Silver,car,...,205.441595,44.850238,4680.761595,True,True,True,True,True,True,True
3,validation,track0061,4,track0061[04].png,data/raw/ufpr_alpr/validation/track0061/track0...,data/raw/ufpr_alpr/validation/track0061/track0...,1920,1080,GoPro Hero4 Silver,car,...,204.934911,43.876639,5154.875648,True,True,True,True,True,True,True
4,validation,track0061,5,track0061[05].png,data/raw/ufpr_alpr/validation/track0061/track0...,data/raw/ufpr_alpr/validation/track0061/track0...,1920,1080,GoPro Hero4 Silver,car,...,202.103896,45.728024,4662.446359,True,True,True,True,True,True,True


In [5]:
# Rename columns for easier analysis
validation_metadata["plate_sharpness"] = (
    validation_metadata["plate_blur_score"]
)

validation_metadata["image_sharpness"] = (
    validation_metadata["image_blur_score"]
)

In [6]:
# Fix image names to match metadata
valid_extensions = {".png", ".jpg", ".jpeg"}

validation_image_paths = sorted(
    path
    for path in VAL_IMAGE_DIR.iterdir()
    if path.suffix.lower() in valid_extensions
)

print("Validation image files:", len(validation_image_paths))

Validation image files: 900


In [7]:
# Measure IoU which is how overlapping two bounding boxes are (one is label from dataset, one is YOLO's prediction)
# 0 is no overlap, 1 is perfect overlap
def calculate_iou(box_a: np.ndarray, box_b: np.ndarray,) -> float:
    """
    Calculate IoU between two boxes in:
    [xmin, ymin, xmax, ymax] format.
    """
    x_left = max(float(box_a[0]), float(box_b[0]))
    y_top = max(float(box_a[1]), float(box_b[1]))
    x_right = min(float(box_a[2]), float(box_b[2]))
    y_bottom = min(float(box_a[3]), float(box_b[3]))

    intersection_width = max(0.0, x_right - x_left)
    intersection_height = max(0.0, y_bottom - y_top)

    intersection_area = intersection_width * intersection_height

    area_a = max(0.0, float(box_a[2] - box_a[0])) * max(
        0.0,
        float(box_a[3] - box_a[1]),
    )

    area_b = max(0.0, float(box_b[2] - box_b[0])) * max(
        0.0,
        float(box_b[3] - box_b[1]),
    )

    union_area = area_a + area_b - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area

In [ ]:
CONFIDENCE_THRESHOLD = 0.25

# top_iou is highest confidence prediction, best_iou is highest IoU among every prediction in the image
def collect_predictions(
    model: YOLO,
    image_paths: list[Path],
    ground_truth: dict,
    image_size: int,
    model_label: str,
    confidence_threshold: float = 0.25,
) -> pd.DataFrame:
    results = model.predict(
        source=[str(path) for path in image_paths],
        imgsz=image_size,
        conf=confidence_threshold,
        device=DEVICE,
        stream=True,
        verbose=False,
        max_det=20,
    )

    records = []

    for result in results:
        image_name = Path(result.path).name

        if image_name not in ground_truth:
            raise KeyError(
                f"No ground-truth metadata for {image_name}"
            )

        gt_values = ground_truth[image_name]

        gt_box = np.array(
            [
                gt_values["plate_xmin"],
                gt_values["plate_ymin"],
                gt_values["plate_xmax"],
                gt_values["plate_ymax"],
            ],
            dtype=float,
        )

        number_of_predictions = len(result.boxes)

        if number_of_predictions == 0:
            records.append(
                {
                    "image_name": image_name,
                    f"detected_{model_label}": False,
                    f"num_predictions_{model_label}": 0,
                    f"top_confidence_{model_label}": 0.0,
                    f"top_iou_{model_label}": 0.0,
                    f"best_iou_{model_label}": 0.0,
                    f"best_confidence_{model_label}": 0.0,
                    f"pred_xmin_{model_label}": np.nan,
                    f"pred_ymin_{model_label}": np.nan,
                    f"pred_xmax_{model_label}": np.nan,
                    f"pred_ymax_{model_label}": np.nan,
                }
            )
            continue

        predicted_boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        confidence_scores = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
        )

        iou_scores = np.array(
            [
                calculate_iou(predicted_box, gt_box)
                for predicted_box in predicted_boxes
            ]
        )

        top_confidence_index = int(
            np.argmax(confidence_scores)
        )

        best_iou_index = int(
            np.argmax(iou_scores)
        )

        top_box = predicted_boxes[top_confidence_index]

        records.append(
            {
                "image_name": image_name,
                f"detected_{model_label}": True,
                f"num_predictions_{model_label}": number_of_predictions,
                f"top_confidence_{model_label}": float(
                    confidence_scores[top_confidence_index]
                ),
                f"top_iou_{model_label}": float(
                    iou_scores[top_confidence_index]
                ),
                f"best_iou_{model_label}": float(
                    iou_scores[best_iou_index]
                ),
                f"best_confidence_{model_label}": float(
                    confidence_scores[best_iou_index]
                ),
                f"pred_xmin_{model_label}": float(top_box[0]),
                f"pred_ymin_{model_label}": float(top_box[1]),
                f"pred_xmax_{model_label}": float(top_box[2]),
                f"pred_ymax_{model_label}": float(top_box[3]),
            }
        )

    prediction_frame = pd.DataFrame(records)

    if len(prediction_frame) != len(image_paths):
        raise ValueError(
            "Prediction row count does not match image count."
        )

    return prediction_frame